 该项目主要是利用大模型微调完成中医药命名实体识别（NER）的任务


首先，我们需要加载package

In [20]:
import os
import json
import torch
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, TaskType, get_peft_model
from modelscope import snapshot_download

数据预处理

In [22]:
import json

def process_data(input_file_path, output_file_path):
    with open(input_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    current_input = []
    current_output = []

    with open(output_file_path, 'w', encoding='utf-8') as outfile:
        for line in lines:
            line = line.strip()
            if line:
                parts = line.split(' ')
                if len(parts) == 2:
                    char, label = parts
                    current_input.append(char)
                    current_output.append(label)
                else:
                    print(f"⚠️ 错误：数据格式不正确，跳过行：'{line}'")
            else:
                # 空行：写入一条记录
                if current_input and current_output:
                    item = {
                        "instruction": "请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，O表示非实体部分。最后以列表的格式输出结果：\n",
                        "input": str(current_input),
                        "output": str(current_output)
                    }
                    json_line = json.dumps(item, ensure_ascii=False)
                    outfile.write(json_line + "\n")
                    current_input = []
                    current_output = []

        # 处理文件末尾（如果没有空行）
        if current_input and current_output:
            item = {
                "instruction": "请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，O表示非实体部分。最后以列表的格式输出结果：\n",
                "input": str(current_input),
                "output": str(current_output)
            }
            json_line = json.dumps(item, ensure_ascii=False)
            outfile.write(json_line + "\n")

    print(f"✅ 处理完成：每条数据已单独写入 {output_file_path}")
    
# 生成处理后的数据集
input_file_path_train = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.train"
output_file_path_train = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical_train.json"
input_file_path_test = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.test"
output_file_path_test = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical_test.json"
input_file_path_dev = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.dev"
output_file_path_dev = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical_dev.json"

process_data(input_file_path_train, output_file_path_train)
process_data(input_file_path_test, output_file_path_test)
process_data(input_file_path_dev, output_file_path_dev)

⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
⚠️ 错误：数据格式不正确，跳过行：'O'
✅ 处理完成：每条数据已单独写入 /root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical_train.json
✅ 处理完成：每条数据已单独写入 /root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical_test.json
✅ 处理完成：每条数据已单独写入 /root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical_dev.json


In [ ]:
def process_data(input_file_path, output_file_path):
    with open(input_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    dataset = []
    current_input = []
    current_output = []

    for line in lines:
        line = line.strip()
        if line:
            parts = line.split(' ')
            if len(parts) == 2:
                char, label = parts
                current_input.append(char)
                current_output.append(label)
            else:
                print(f"错误：数据格式不正确，跳过行：'{line}'")
        else:
            if current_input and current_output:
                dataset.append({
                    "instruction": "请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，O表示非实体部分。最后以列表的格式输出结果：\n",
                    "input": str(current_input),
                    "output": str(current_output)
                })
                # 重置当前组数据
                current_input = []
                current_output = []

    # 处理最后一组数据（如果没有空行结尾）
    if current_input and current_output:
        dataset.append({
            "instruction": "请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，O表示非实体部分。最后以列表的格式输出结果：\n",
            "input": str(current_input),
            "output": str(current_output)
        })

    # 保存数据
    with open(output_file_path, 'w', encoding='utf-8') as file:
        json.dump(dataset, file, ensure_ascii=False, indent=2)

    print(f"处理完成，新的数据集已保存到 {output_file_path}")

# 生成处理后的数据集
input_file_path_train = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.train"
output_file_path_train = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical.train"
input_file_path_test = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.test"
output_file_path_test = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical.test"
input_file_path_dev = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.dev"
output_file_path_dev = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical.dev"

process_data(input_file_path_train, output_file_path_train)
process_data(input_file_path_test, output_file_path_test)
process_data(input_file_path_dev, output_file_path_dev)

加载模型

In [ ]:
def load_model(model_dir, device='cuda'):
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype="auto",
        trust_remote_code=True
        # device_map="auto"     
    )
    
    model = model.to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    return model, tokenizer

cache_dir = '/root/data2/anti_fraud/models/modelscope/hub'
model_dir = snapshot_download('Qwen/Qwen2-7B', cache_dir=cache_dir, revision='master')
model7b, tokenizer = load_model(model_dir, device='cuda')

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

构建数据集

In [18]:
import ast

# load the json file
def load_jsonl(path):
    # fine tune every single line
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)   # data will be a Python list
        
    return data
    
def preprocess(item, tokenizer, max_length=2048):
    system_message = "You are a helpful assistant."
    user_message = item['instruction'] + item['input']
    assistant_message = json.dumps({"name":item["output"]}, ensure_ascii=False)
    
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(f"<|im_start|>system\n{system_message}<|im_end|>\n<|im_start|>user\n{user_message}<|im_end|>\n<|im_start|>assistant\n", add_special_tokens=False)  
    response = tokenizer(assistant_message, add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]  
    # -100是一个特殊的标记，用于指示指令部分的token不应参与损失计算
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]  
    
    # 对输入长度做一个限制保护，超出截断
    return {
        "input_ids": input_ids[:max_length],
        "attention_mask": attention_mask[:max_length],
        "labels": labels[:max_length]
    }
    
def load_dataset(train_path, tokenizer):
    train_df = load_jsonl(train_path)
    df = pd.DataFrame(train_df)
    df["input"] = df["input"].apply(ast.literal_eval)
    df["output"] = df["output"].apply(ast.literal_eval)
    print(df)
    train_dataset = df.map(lambda x: preprocess(x, tokenizer), remove_columns=df.columns)

    return train_dataset

In [19]:
train_path = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical.dev"

train_dataset = load_dataset(train_path, tokenizer) # tokenizer is defined in the prev block

print(train_dataset)

                                         instruction  \
0  请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，...   
1  请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，...   

                                               input  \
0  [投, 活, 络, 效, 灵, 丹, 加, 味, ：, 当, 归, 、, 丹, 参, 各, ...   
1  [目, 的, 补, 气, 健, 脾, 升, 清, 法, 治, 疗, 糖, 尿, 病, 性, ...   

                                              output  
0  [O, B-方剂, I-方剂, I-方剂, I-方剂, I-方剂, O, O, O, B-中...  
1  [O, O, O, O, O, O, O, O, O, O, O, B-西医诊断, I-西医...  


TypeError: load_dataset.<locals>.<lambda>() got an unexpected keyword argument 'remove_columns'

Load the model and the peft model

In [ ]:
def load_model(model_path, device='cuda'):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16)
    model.enable_input_require_grads() # 开启梯度检查点时，要执行该方法
    return model.to(device)

model = load_model(model_path, device)
model

# LoRA model
def build_peft_model(model):
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, 
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        inference_mode=False, # 训练模式
        r=8, 
        lora_alpha=16,   
        lora_dropout=0.05
    )
    return get_peft_model(model, config)

peft_model = build_peft_model(model)
peft_model

# Option2: QLoRA model

Set up the training arguments

In [ ]:
def build_train_arguments(output_path):
    return TrainingArguments(
        output_dir=output_path,
        per_device_train_batch_size=4,  # 每个设备（如每个GPU）的训练批次大小
        gradient_accumulation_steps=4,  # 梯度累积的步骤数，相当于增大批次大小
        logging_steps=10,                
        num_train_epochs=3,    
        eval_strategy="steps",  
        eval_steps=10, # 设置评估的步数，与保存步数一致
        save_steps=10, # 为了快速演示，这里设置20，建议设置成100
        learning_rate=1e-4,
        save_on_each_node=True,
        load_best_model_at_end=True, # 在训练结束时加载最佳模型
        gradient_checkpointing=True  #  启用梯度检查点以节省内存
    )
    
def build_trainer(model, tokenizer, args, train_dataset, eval_dataset):
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # 早停回调
    )
    
trainer = build_trainer(peft_model, tokenizer, build_train_arguments(output_path), train_dataset, eval_dataset)
trainer.train()

### 推理训练好的模型